# CropFusion — Train (end-to-end pipeline)

Kaggle GPU notebook for the Training Platform — runs the full R2.3 chain:
bootstrap, frozen corpus resolution (R5.2.7 supervised crop data) and training
(`run_pipeline.py`), then records the latest checkpoint into
`training/kaggle/checkpoints/metadata.json` + a `checkpoint.json` report so
downstream stages can find it.

- Dataset (attach this notebook to it): `shathanandabhatn/crop-yield-forecasting-karnataka-dakshina-kannada`
- Reports land under `training/kaggle/outputs/reports/`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Brijesh2005/CropPrep.git'
REPO_ROOT = Path('/kaggle/working/CropPrep')

if not (REPO_ROOT / '.git').exists():
    print(f'cloning CropPrep -> {REPO_ROOT}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')


## 1.1 P100 GPU fix

Kaggle's base PyTorch (cu128) dropped Pascal `sm_60` kernels, so the P100
cannot execute any kernel (`CUDA error: no kernel image is available`).
Reinstall torch from the cu126 index, which still ships `sm_60` cubins
(verified fix, kaggle/docker-python#1546).


In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'],
    check=True,
)
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| arch', torch.cuda.get_arch_list())


## 1. Bootstrap environment + data sources

Verifies the Kaggle runtime, Python, CUDA and GPU; installs the editable
packages; creates the workspace; reports the Dataset Manager provider
manifests; verifies repository integrity and tabular datasets; generates the
environment / GPU / dependency / storage / workspace / configuration reports.
Pass `--ensure-data` only when the imagery is **not** attached as a Kaggle
dataset.

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install

## 2. Pipeline orchestration readiness

Initialises every pipeline component (Dataset Manager, STAM, preprocessing,
trainer, evaluator, exporter) and writes the orchestration report. This is a
dry run — nothing is trained.

In [ ]:
!python training/kaggle/scripts/run_training.py

## 3. System check

Runs the Training Validator (config / python / GPU / dependencies / folders /
disk / providers) and writes the validation report. Exit code 0 means ready.

In [ ]:
!python training/kaggle/scripts/system_check.py

## 4. Full pipeline (corpus + training)

Runs the end-to-end training driver: Dataset Manager imagery, STAM,
frozen corpus (R5.2.7 supervised crop data) or ObservationResolver corpus,
then the Experiment (preprocess -> model -> trainer -> evaluate). Writes
`pipeline.json` + `frozen_corpus.json` (or `corpus.json`) under
`training/kaggle/outputs/reports/`.

The frozen corpus path (`--frozen-crop-csv`) bypasses ObservationResolver
and uses the pre-validated R5.2.7 supervised crop data with exact class
counts and spatial split (Bantwal val, Sullia test).

In [ ]:
!python training/kaggle/scripts/run_pipeline.py --frozen-crop-csv govt_crop_matched_v1/crop_supervised_v1.csv --frozen-manifest training_manifests/crop_supervised_v1_manifest.json; echo "run_pipeline_exit=$?"

## 5. Checkpoint report

Locates the latest checkpoint under the run directory (by modification time),
registers it in the workspace checkpoint metadata registry and writes
`training/kaggle/outputs/reports/checkpoint.json` so the orchestrator can
download it for the export stage.

In [ ]:
import json
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/CropPrep')
report_path = REPO_ROOT / 'training/kaggle/outputs/reports/pipeline.json'
info = {'found': False}

if report_path.exists():
    pipeline = json.loads(report_path.read_text())
    training = pipeline.get('training', {})
    if training.get('status') == 'completed' and training.get('run_dir'):
        run_dir = Path(training['run_dir'])
        ckpt_dir = run_dir / 'checkpoints'
        candidates = sorted(
            [p for p in ckpt_dir.rglob('*.pt')] if ckpt_dir.exists() else [],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            latest = candidates[0]
            from training.kaggle.config import load_paths_config, WorkspaceLayout
            from training.kaggle.workspace import WorkspaceManager
            paths = load_paths_config()
            layout = WorkspaceLayout.resolve(paths, repo_root=REPO_ROOT)
            workspace = WorkspaceManager(layout)
            workspace.create()
            report = training.get('report', {})
            entry = workspace.checkpoints.register(
                run_name=report.get('run_name') or run_dir.name,
                stage='best',
                metrics=report.get('evaluation', {}) or report.get('training', {}),
                path=str(latest),
                resume=True,
            )
            info = {
                'found': True,
                'path': str(latest),
                'repo_relative': str(latest.relative_to(REPO_ROOT)),
                'run_dir': str(run_dir),
                'registered': entry,
            }
            print('latest checkpoint:', latest)
        else:
            print('no checkpoint files under', ckpt_dir)
    else:
        print('training status:', training.get('status'), '-', training.get('reason', ''))
else:
    print('pipeline report missing:', report_path)

out = REPO_ROOT / 'training/kaggle/outputs/reports/checkpoint.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(info, indent=2), encoding='utf-8')
print('checkpoint report ->', out)

## 6. Release sources

Persists the train-side artefacts the export stage needs to assemble the
Prediction Platform release package: fitted scaler + label encoder, dataset
snapshot (`metadata.db`, historical_context / location_index /
village_metadata parquets), metrics and a `sources.json` manifest, under
`training/artifacts/release_sources`. The orchestrator uploads these with the
checkpoint dataset so the export kernel can mount them.

In [ ]:
!python training/kaggle/scripts/package_sources.py; echo "package_sources_exit=$?"